In [1]:
import pandas as pd


In [2]:
acs2024 = pd.read_csv("/Users/carmenk/Documents/CSS/Capstone/acs2024_csv_5year/psam_pusa.csv")

In [3]:
acs2024.head()

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,STATE,ADJINC,PWGTP,AGEP,...,PWGTP71,PWGTP72,PWGTP73,PWGTP74,PWGTP75,PWGTP76,PWGTP77,PWGTP78,PWGTP79,PWGTP80
0,P,2020GQ0000060,6,1,1301,3,1,1222017,11,68,...,11,21,1,10,21,20,10,12,11,10
1,P,2020GQ0000084,6,1,1403,3,1,1222017,9,18,...,9,9,1,17,9,9,9,1,17,0
2,P,2020GQ0000128,6,1,1801,3,1,1222017,3,35,...,3,0,4,4,0,4,0,7,7,3
3,P,2020GQ0000189,6,1,1501,3,1,1222017,13,46,...,13,26,13,14,2,2,26,24,2,23
4,P,2020GQ0000207,6,1,700,3,1,1222017,41,79,...,3,82,76,44,49,3,37,41,40,36


In [4]:
print("Columns:\n",acs2024.columns.tolist())

Columns:
 ['RT', 'SERIALNO', 'DIVISION', 'SPORDER', 'PUMA', 'REGION', 'STATE', 'ADJINC', 'PWGTP', 'AGEP', 'CIT', 'CITWP', 'COW', 'DDRS', 'DEAR', 'DEYE', 'DOUT', 'DPHY', 'DRAT', 'DRATX', 'DREM', 'ENG', 'FER', 'GCL', 'GCM', 'GCR', 'HINS1', 'HINS2', 'HINS3', 'HINS4', 'HINS5', 'HINS6', 'HINS7', 'INTP', 'JWMNP', 'JWRIP', 'JWTRNS', 'LANX', 'MAR', 'MARHD', 'MARHM', 'MARHT', 'MARHW', 'MARHYP', 'MIG', 'MIL', 'MLPA', 'MLPB', 'MLPCD', 'MLPE', 'MLPFG', 'MLPHJ', 'MLPIK', 'NWAB', 'NWAV', 'NWLA', 'NWLK', 'NWRE', 'OIP', 'PAP', 'RELSHIPP', 'RETP', 'SCH', 'SCHG', 'SCHL', 'SEMP', 'SEX', 'SSIP', 'SSP', 'WAGP', 'WKHP', 'WKL', 'WKWN', 'WRK', 'YOEP', 'ANC', 'ANC1P', 'ANC2P', 'DECADE', 'DIS', 'DRIVESP', 'ESP', 'ESR', 'FOD1P', 'FOD2P', 'HICOV', 'HISP', 'INDP', 'JWAP', 'JWDP', 'LANP', 'MIGPUMA', 'MIGSP', 'MSP', 'NAICSP', 'NATIVITY', 'NOP', 'OC', 'OCCP', 'PAOC', 'PERNP', 'PINCP', 'POBP', 'POVPIP', 'POWPUMA', 'POWSP', 'PRIVCOV', 'PUBCOV', 'QTRBIR', 'RAC1P', 'RAC2P19', 'RAC2P24', 'RAC3P', 'RACAIAN', 'RACASN', 'RAC

# ACS 2020–2024 5-Year PUMS: Variable Extraction 

## Variable Codebook Mapping

| Concept | ACS PUMS Code | Description |
|---|---|---|
| **Gender** | `SEX` | 1 = Male, 2 = Female |
| **Race** | `RAC1P` | 1=White, 2=Black, 3=AIAN, 6=Asian, 7=NHOPI, 8=Other, 9=Two+ races (see below) |
| **Education** | `SCHL` | Attainment recode → `educ_category` (see below) |
| **State** | `STATE` | FIPS state code (51 states incl. DC) |
| **Census Division (region9)** | `DIVISION` | 1–9 (9 Census divisions) |
| **Census Region** | `REGION` | 1–4 (4 regions: NE, MW, S, W) |
| **% Driving alone** | `JWTRNS` | Means of transportation; value 01 = drove alone |
| **% Same-sex households** | `RELSHIPP` + `SEX` | RELSHIPP 23 = same-sex spouse, 24 = same-sex unmarried partner |
| **CO₂ emissions** | *external* | EPA GHGRP or EIA state-level data — not in ACS |
| **Dem Presidential vote share** | *external* | MIT Election Lab or Dave Leip's Atlas — not in ACS |

### SCHL → `educ_category` recode (4 categories, matches CCAM):
| SCHL values | Code | Category |
|---|---|---|
| 01–15 | 1 | Less than HS |
| 16–17 | 2 | High school |
| 18–20 | 3 | Some college (incl. Associate's) |
| 21–24 | 4 | Bachelor's degree or higher |

### RAC1P labels:
`1=White alone, 2=Black/AA alone, 3=AIAN alone, 4=Alaska Native alone, 5=AIAN tribes, 6=Asian alone, 7=NHOPI alone, 8=Other alone, 9=Two or more races`

### DIVISION (region9) labels:
`1=New England, 2=Mid-Atlantic, 3=E. North Central, 4=W. North Central, 5=South Atlantic, 6=E. South Central, 7=W. South Central, 8=Mountain, 9=Pacific`


In [ ]:
import pandas as pd
import numpy as np

DATA_DIR = "/Users/carmenk/Documents/CSS/Capstone/acs2024_csv_5year/"
KEEP_COLS = ["SERIALNO", "SPORDER", "PWGTP", "STATE", "DIVISION", "REGION",
             "SEX", "RAC1P", "SCHL", "JWTRNS", "RELSHIPP"]

# Load all four person files
parts = []
for suffix in ["a", "b", "c", "d"]:
    path = f"{DATA_DIR}psam_pus{suffix}.csv"
    df_part = pd.read_csv(path, usecols=KEEP_COLS, low_memory=False)
    parts.append(df_part)
    print(f"Loaded psam_pus{suffix}.csv  → {len(df_part):,} rows")

acs = pd.concat(parts, ignore_index=True)
print(f"\nTotal records: {len(acs):,}")
print(f"Unique states: {acs['STATE'].nunique()}")


In [ ]:
# ── Recode variables ──────────────────────────────────────────────────────────

# gender
acs["gender"] = acs["SEX"].map({1: "Male", 2: "Female"})

# race (RAC1P)
race_map = {
    1: "White alone",
    2: "Black/AA alone",
    3: "AIAN alone",
    4: "Alaska Native alone",
    5: "AIAN tribes",
    6: "Asian alone",
    7: "NHOPI alone",
    8: "Other alone",
    9: "Two or more races",
}
acs["race"] = acs["RAC1P"].map(race_map)

# educ_category — 4 categories matching CCAM
# SCHL 01–15 → 1 Less than HS
# SCHL 16–17 → 2 High school
# SCHL 18–20 → 3 Some college (incl. Associate's)
# SCHL 21–24 → 4 Bachelor's degree or higher
def schl_to_educ(v):
    if pd.isna(v):
        return np.nan
    v = int(v)
    if v <= 15: return 1
    if v <= 17: return 2
    if v <= 20: return 3
    return 4

acs["educ_category"] = acs["SCHL"].apply(schl_to_educ)

educ_labels = {
    1: "Less than HS",
    2: "High school",
    3: "Some college",
    4: "Bachelor's or higher",
}

# region9 (DIVISION labels)
division_map = {
    1: "New England",
    2: "Mid-Atlantic",
    3: "E. North Central",
    4: "W. North Central",
    5: "South Atlantic",
    6: "E. South Central",
    7: "W. South Central",
    8: "Mountain",
    9: "Pacific",
}
acs["region9"] = acs["DIVISION"].map(division_map)

# drove_alone flag — NA where person has no commute (JWTRNS is NaN)
acs["drove_alone"] = (acs["JWTRNS"] == 1).where(acs["JWTRNS"].notna()).astype("Int8")

# same_sex_hh flag: RELSHIPP 23 (same-sex spouse) or 24 (same-sex partner)
acs["same_sex_hh"] = acs["RELSHIPP"].isin([23, 24]).astype(int)

print("Recoding complete. Sample:")
print(acs[["SCHL", "educ_category", "SEX", "gender", "RAC1P", "race",
           "DIVISION", "region9"]].head(8).to_string())


Recoding complete. Sample:
   SCHL  educ_category  SEX  gender  RAC1P               race  DIVISION           region9
0  16.0            2.0    1    Male      1        White alone         6  E. South Central
1  18.0            3.0    2  Female      2     Black/AA alone         6  E. South Central
2  16.0            2.0    2  Female      1        White alone         6  E. South Central
3  18.0            3.0    2  Female      9  Two or more races         6  E. South Central
4  22.0            4.0    2  Female      1        White alone         6  E. South Central
5  20.0            3.0    2  Female      1        White alone         6  E. South Central
6  16.0            2.0    1    Male      2     Black/AA alone         6  E. South Central
7  16.0            2.0    2  Female      2     Black/AA alone         6  E. South Central


In [ ]:
# ── Descriptive Statistics ────────────────────────────────────────────────────

# 1. Gender distribution
print("=" * 60)
print("GENDER  (SEX)")
print("=" * 60)
g = acs["gender"].value_counts(normalize=True).mul(100).round(2)
print(g.to_string(), "\n")

# 2. Race distribution
print("=" * 60)
print("RACE  (RAC1P)")
print("=" * 60)
r = acs["race"].value_counts(normalize=True).mul(100).round(2)
print(r.to_string(), "\n")

# 3. Education category (numeric codes + labels)
print("=" * 60)
print("EDUCATION CATEGORY  (SCHL recoded, 4 categories)")
print("=" * 60)
e = (acs["educ_category"]
     .map(educ_labels)
     .value_counts(normalize=True)
     .mul(100).round(2))
# reorder by numeric code
e = e.reindex([educ_labels[k] for k in sorted(educ_labels)])
print(e.to_string(), "\n")

# 4. State coverage
print("=" * 60)
print("STATE  (FIPS codes)")
print("=" * 60)
print(f"  Unique states in data: {acs['STATE'].nunique()}")
print(f"  Min FIPS: {acs['STATE'].min()}  Max FIPS: {acs['STATE'].max()}")
print(f"  Records per state (top 10):")
print(acs["STATE"].value_counts().head(10).to_string(), "\n")

GENDER  (SEX)
gender
Female    50.85
Male      49.15 

RACE  (RAC1P)
race
White alone            66.98
Two or more races      10.93
Black/AA alone          8.93
Asian alone             6.06
Other alone             5.66
AIAN alone              1.02
AIAN tribes             0.18
NHOPI alone             0.17
Alaska Native alone     0.06 

EDUCATION CATEGORY  (SCHL recoded, 4 categories)
educ_category
Less than HS            24.89
High school             22.85
Some college            24.33
Bachelor's or higher    27.93 

STATE  (FIPS codes)
  Unique states in data: 51
  Min FIPS: 1  Max FIPS: 56
  Records per state (top 10):
STATE
6     1867063
48    1326295
36     984474
12     982868
42     660142
17     627720
39     586936
37     515641
26     505689
13     493651 



In [ ]:
# 5. Census Division / region9
print("=" * 60)
print("REGION9  (DIVISION — 9 Census Divisions)")
print("=" * 60)
d = acs["region9"].value_counts(normalize=True).mul(100).round(2)
print(d.to_string(), "\n")

# 6. % Driving alone (workers only)
print("=" * 60)
print("% DRIVING ALONE  (JWTRNS == 1, among workers with non-null JWTRNS)")
print("=" * 60)
respondents = acs["JWTRNS"].notna()
pct_drive = acs.loc[respondents, "drove_alone"].mean() * 100
print(f"  Workers with commute data: {workers.sum():,}")
print(f"  % drove alone:             {pct_drive:.1f}%")
drive_state = acs[workers].groupby("STATE")["drove_alone"].mean().mul(100).round(1)
print(f"  State-level range: {drive_state.min()}% – {drive_state.max()}%\n")

# 7. % Same-sex households
print("=" * 60)
print("% SAME-SEX HOUSEHOLDS  (RELSHIPP ∈ {23, 24})")
print("=" * 60)
pct_ss = acs["same_sex_hh"].mean() * 100
print(f"  % persons in same-sex couple (national): {pct_ss:.3f}%")
ss_state = acs.groupby("STATE")["same_sex_hh"].mean().mul(100).round(3)
print(f"  State-level range: {ss_state.min()}% – {ss_state.max()}%")
print(ss_state.describe().round(4).to_string(), "\n")

# 8. Note on external variables
print("=" * 60)
print("POINT-SOURCE CO₂ EMISSIONS & DEM PRESIDENTIAL VOTE SHARE")
print("=" * 60)
print("  Not in ACS. Suggested sources:")
print("  • CO₂:           EPA GHGRP / EIA  →  https://www.epa.gov/ghgreporting")
print("  • Dem vote share: MIT Election Lab →  https://electionlab.mit.edu/data")
print("  Merge to ACS state-level aggregates on STATE FIPS.")


REGION9  (DIVISION — 9 Census Divisions)
region9
South Atlantic      19.49
Pacific             15.99
E. North Central    14.69
Mid-Atlantic        12.98
W. South Central    11.69
Mountain             7.68
W. North Central     6.89
E. South Central     5.92
New England          4.67 

% DRIVING ALONE  (JWTRNS == 1, among workers with non-null JWTRNS)
  Workers with commute data: 7,303,261
  % drove alone:             77.1%
  State-level range: 29.2% – 90.4%

% SAME-SEX HOUSEHOLDS  (RELSHIPP ∈ {23, 24})
  % persons in same-sex couple (national): 0.384%
  State-level range: 0.132% – 1.624%
count    51.0000
mean      0.3809
std       0.2166
min       0.1320
25%       0.2600
50%       0.3380
75%       0.4445
max       1.6240 

POINT-SOURCE CO₂ EMISSIONS & DEM PRESIDENTIAL VOTE SHARE
  Not in ACS. Suggested sources:
  • CO₂:           EPA GHGRP / EIA  →  https://www.epa.gov/ghgreporting
  • Dem vote share: MIT Election Lab →  https://electionlab.mit.edu/data
  Merge to ACS state-level aggreg

# Household Income in ACS PUMS

## Key finding
**Household income (`HINCP`) lives in the *housing-unit* files (`psam_husa/b/c/d.csv`), which are not in this folder.**  
Only the *person-level* files are present here.

### Person-level income variables available:
| Code | Description | Type |
|---|---|---|
| **`PINCP`** | Total person income (past 12 months) | Continuous $ |
| `PERNP` | Total person earnings (wages + self-employment) | Continuous $ |
| `WAGP` | Wages or salary | Continuous $ |
| `SEMP` | Self-employment income | Continuous $ |
| `RETP` | Retirement income | Continuous $ |
| **`POVPIP`** | Income-to-poverty-level ratio (%) | 0–501 (501 = 501%+) |

### Income adjustment
All dollar amounts must be **inflation-adjusted** using `ADJINC`:
```
adjusted_income = PINCP * (ADJINC / 1_000_000)
```
`ADJINC` is a 7-digit factor (e.g. 1222017 → multiplier = 1.222). It aligns all survey years in the 5-year file (2020–2024) to a common dollar base.

### PINCP value range (from data):
- Can be **negative** (business losses)
- `NaN` for children under 15 or group-quarters
- Continuous scale (no fixed categories in ACS — you recode yourself)

### Standard `income_category` recode matching CCAM brackets:
| PINCP range (adj.) | income_category |
|---|---|
| < $30,000 | "Less than $30,000" |
| $30,000–$49,999 | "$30,000–$49,999" |
| $50,000–$74,999 | "$50,000–$74,999" |
| $75,000–$99,999 | "$75,000–$99,999" |
| ≥ $100,000 | "$100,000 or more" |

> **For true household income**: download housing-unit PUMS files (`psam_hus*.csv`) from census.gov and merge on `SERIALNO`.


In [ ]:
# ── PINCP → income_category recode ───────────────────────────────────────────

# Inflation-adjust to common dollar base
acs_inc = pd.read_csv(
    "/Users/carmenk/Documents/CSS/Capstone/acs2024_csv_5year/psam_pusa.csv",
    usecols=["SERIALNO", "SPORDER", "PWGTP", "STATE", "PINCP", "ADJINC"],
    low_memory=False
)

# Apply ADJINC factor (7-digit, divide by 1,000,000)
acs_inc["pincp_adj"] = acs_inc["PINCP"] * (acs_inc["ADJINC"] / 1_000_000)

# Recode to income_category (CCAM-compatible brackets)
inc_bins  = [-float("inf"), 30000, 50000, 75000, 100000, float("inf")]
inc_labels = ["Less than $30,000", "$30,000–$49,999",
              "$50,000–$74,999",   "$75,000–$99,999", "$100,000 or more"]

acs_inc["income_category"] = pd.cut(
    acs_inc["pincp_adj"],
    bins=inc_bins,
    labels=inc_labels,
    right=False
)

# Descriptive stats
print("=== PINCP (raw, unadjusted) ===")
print(acs_inc["PINCP"].describe().round(0).to_string())

print("\n=== pincp_adj (inflation-adjusted $) ===")
print(acs_inc["pincp_adj"].describe().round(0).to_string())

print("\n=== income_category distribution ===")
dist = acs_inc["income_category"].value_counts(sort=False, normalize=True).mul(100).round(2)
print(dist.to_string())

print(f"\nNull / not applicable (children <15, GQ): {acs_inc['income_category'].isna().sum():,}")


=== PINCP (raw, unadjusted) ===
count    4052193.0
mean       52020.0
std        80890.0
min       -11500.0
25%         9000.0
50%        30000.0
75%        65000.0
max      1945000.0

=== pincp_adj (inflation-adjusted $) ===
count    4052193.0
mean       57566.0
std        89341.0
min       -12220.0
25%         9943.0
50%        32639.0
75%        71594.0
max      1974661.0

=== income_category distribution ===
income_category
Less than $30,000    47.37
$30,000–$49,999      15.91
$50,000–$74,999      13.26
$75,000–$99,999       7.81
$100,000 or more     15.65

Null / not applicable (children <15, GQ): 696,463


# County-Level Poststratification Frame

Build an MRP poststratification frame at **county** resolution with the structure:

`county_fips × gender × race4 × educ_category → N (weighted adult pop count)`

plus county-level covariates: `co2_per_capita` and `dem_share_two_party`.

### Key constraint: ACS PUMS has no county identifier
PUMS records carry `STATE` + `PUMA` (Public Use Microdata Area, ≥ 100k pop).
To reach counties we use the **2020 Census PUMA-to-county geographic relationship file**
which gives each PUMA's area overlap with every county it intersects.
Each PUMS record's weight is split across counties proportional to that area overlap.

In [ ]:
import pandas as pd
import numpy as np
import urllib.request, os

DATA_DIR = "/Users/carmenk/Documents/CSS/Capstone/data/"
ACS_DIR  = "/Users/carmenk/Documents/CSS/Capstone/acs2024_csv_5year/"

# ── Load county-level covariates ──────────────────────────────────────────────
carbon = pd.read_csv(DATA_DIR + "carbon_county.csv",
                     dtype={"county_fips": str, "state_fips": str})
pres   = pd.read_csv(DATA_DIR + "pres_county.csv",
                     dtype={"county_fips": str, "state_fips": str})

carbon["county_fips"] = carbon["county_fips"].str.zfill(5)
pres["county_fips"]   = pres["county_fips"].str.zfill(5)

covars = (
    carbon[["county_fips", "state_fips", "county_name", "state_abbr", "co2_per_capita"]]
    .merge(pres[["county_fips", "dem_share_two_party"]], on="county_fips", how="outer")
)

print(f"Carbon: {len(carbon):,} counties | Pres: {len(pres):,} counties")
print(f"Merged: {len(covars):,} | missing co2: {covars['co2_per_capita'].isna().sum()} "
      f"| missing dem_share: {covars['dem_share_two_party'].isna().sum()}")
print("\nco2_per_capita:");      print(covars["co2_per_capita"].describe().round(2).to_string())
print("\ndem_share_two_party:"); print(covars["dem_share_two_party"].describe().round(3).to_string())

Carbon: 3,144 counties | Pres: 3,145 counties
Merged: 3,191 | missing co2: 47 | missing dem_share: 46

co2_per_capita:
count     3144.00
mean        47.70
std        546.74
min          2.31
25%          8.95
50%         13.29
75%         25.89
max      29420.24

dem_share_two_party:
count    3145.000
mean        0.324
std         0.159
min         0.035
25%         0.200
50%         0.289
75%         0.414
max         0.933


In [ ]:
# ── County FIPS → name mapping ────────────────────────────────────────────────
# Already available from carbon_county.csv — no separate download needed
county_names = (
    carbon[["county_fips", "county_name", "state_abbr", "state_fips"]]
    .drop_duplicates("county_fips")
    .sort_values("county_fips")
    .reset_index(drop=True)
)
print(f"County reference table: {len(county_names):,} counties")
print(county_names.head(8).to_string(index=False))

# ── PUMA-to-county crosswalk via Census Tract → PUMA relationship file ────────
# Strategy: each census tract belongs to exactly one PUMA and one county.
# Count tracts per PUMA-county pair; use tract count as population proxy for afact.
# File: tab-separated, columns STATEFP, COUNTYFP, TRACTCE, PUMA5CE (or *20 suffix)
TRACT_PUMA_PATH = DATA_DIR + "tract_to_puma_2020.txt"
TRACT_PUMA_URL  = ("https://www2.census.gov/geo/docs/maps-data/data/rel2020/"
                   "2020_Census_Tract_to_2020_PUMA.txt")

if not os.path.exists(TRACT_PUMA_PATH):
    print("\nDownloading Census Tract → PUMA crosswalk (~4 MB)...")
    try:
        urllib.request.urlretrieve(TRACT_PUMA_URL, TRACT_PUMA_PATH)
        print(f"Saved → {TRACT_PUMA_PATH}")
    except Exception as e:
        print(f"Download failed: {e}")
        print(
            "\nManual download instructions:\n"
            "  1. Go to: https://www.census.gov/geographies/reference-files/"
            "time-series/geo/relationship-files.html\n"
            "  2. Under '2020 Census', download "
            "'2020 Census Tract to 2020 PUMA'\n"
            f"  3. Save to: {TRACT_PUMA_PATH}"
        )
        raise
else:
    print(f"\nUsing cached: {TRACT_PUMA_PATH}")

tracts = pd.read_csv(TRACT_PUMA_PATH, dtype=str)
print("Raw columns:", tracts.columns.tolist())

# Normalize column names — handle both "STATEFP" and "STATEFP20" variants
tracts.columns = [c.replace("20", "") for c in tracts.columns]

tracts["state_fips"]  = tracts["STATEFP"]
tracts["county_fips"] = tracts["STATEFP"] + tracts["COUNTYFP"]   # 5-digit FIPS
tracts["puma_code"]   = tracts["PUMA5CE"].str.zfill(5)

# Allocation factor = fraction of PUMA's tracts that fall in each county
puma_county = (
    tracts.groupby(["state_fips", "puma_code", "county_fips"])
    .size()
    .reset_index(name="n_tracts")
)
puma_county["afact"] = (
    puma_county["n_tracts"] /
    puma_county.groupby(["state_fips", "puma_code"])["n_tracts"].transform("sum")
)
xwalk = puma_county[["state_fips", "puma_code", "county_fips", "afact"]].copy()

print(f"\nCrosswalk: {len(xwalk):,} PUMA–county pairs | {xwalk['county_fips'].nunique():,} counties")
print(xwalk.head(6).to_string(index=False))

County reference table: 3,144 counties
county_fips    county_name state_abbr state_fips
      01001 Autauga County         AL         01
      01003 Baldwin County         AL         01
      01005 Barbour County         AL         01
      01007    Bibb County         AL         01
      01009  Blount County         AL         01
      01011 Bullock County         AL         01
      01013  Butler County         AL         01
      01015 Calhoun County         AL         01

Saved → /Users/carmenk/Documents/CSS/Capstone/data/tract_to_puma_2020.txt
Raw columns: ['STATEFP', 'COUNTYFP', 'TRACTCE', 'PUMA5CE']

Crosswalk: 4,701 PUMA–county pairs | 3,222 counties
state_fips puma_code county_fips    afact
        01     00100       01033 0.288462
        01     00100       01059 0.211538
        01     00100       01077 0.500000
        01     00200       01083 1.000000
        01     00300       01079 0.261905
        01     00300       01103 0.738095


In [ ]:
# ── Load ACS 5-year PUMS — only model variables + PUMA ───────────────────────
PUMS_COLS = ["PWGTP", "STATE", "PUMA", "DIVISION", "SEX", "RAC1P", "HISP", "SCHL", "AGEP"]

parts = []
for suffix in ["a", "b", "c", "d"]:
    chunk = pd.read_csv(f"{ACS_DIR}psam_pus{suffix}.csv",
                        usecols=PUMS_COLS, low_memory=False)
    parts.append(chunk)
    print(f"  psam_pus{suffix}.csv → {len(chunk):,} rows")

pums = pd.concat(parts, ignore_index=True)

# Adults only (survey population)
pums = pums[pums["AGEP"] >= 18].copy()
print(f"\nAdults 18+: {len(pums):,}")

# gender
pums["gender"] = pums["SEX"].map({1: "Male", 2: "Female"})

# race4 (HISP-first, vectorized)
hisp = pd.to_numeric(pums["HISP"], errors="coerce")
pums["race4"] = np.select(
    [hisp > 1,
     (hisp <= 1) & (pums["RAC1P"] == 1),
     (hisp <= 1) & (pums["RAC1P"] == 2)],
    ["Hispanic", "White", "Black"],
    default="Other"
)

# educ_category  (0,15]=1  (15,17]=2  (17,20]=3  (20,24]=4
pums["educ_category"] = pd.cut(
    pd.to_numeric(pums["SCHL"], errors="coerce"),
    bins=[0, 15, 17, 20, 24], labels=[1, 2, 3, 4]
).astype("Int8")

# zero-pad state and PUMA for crosswalk merge
pums["state_fips"] = pums["STATE"].astype(str).str.zfill(2)
pums["puma_code"]  = pums["PUMA"].astype(str).str.zfill(5)

NameError: name 'pd' is not defined

In [ ]:
# ── Aggregate to PUMA cells → allocate to counties → merge covariates ────────
educ_labels = {1: "Less than HS", 2: "High school",
               3: "Some college",  4: "Bachelor's or higher"}
division_map = {
    1: "New England",    2: "Mid-Atlantic",      3: "E. North Central",
    4: "W. North Central", 5: "South Atlantic",  6: "E. South Central",
    7: "W. South Central", 8: "Mountain",        9: "Pacific",
}

valid = pums.dropna(subset=["gender", "race4", "educ_category"])

# Step 1: sum weights within each PUMA × demographic cell
puma_cells = (
    valid
    .groupby(["state_fips", "puma_code", "DIVISION", "gender", "race4", "educ_category"],
             observed=True)["PWGTP"]
    .sum()
    .reset_index()
    .rename(columns={"PWGTP": "N_puma"})
)
print(f"PUMA-level cells: {len(puma_cells):,}")

# Step 2: join crosswalk and distribute weight by afact
allocated = puma_cells.merge(
    xwalk[["state_fips", "puma_code", "county_fips", "afact"]],
    on=["state_fips", "puma_code"], how="left"
)
n_unmapped = allocated["county_fips"].isna().sum()
if n_unmapped:
    print(f"PUMA cells without county mapping (dropped): {n_unmapped:,}")
    allocated = allocated.dropna(subset=["county_fips"])

allocated["N"] = allocated["N_puma"] * allocated["afact"]

# Step 3: aggregate to county level
poststrat_county = (
    allocated
    .groupby(["county_fips", "state_fips", "DIVISION", "gender", "race4", "educ_category"],
             observed=True)["N"]
    .sum()
    .reset_index()
)
poststrat_county["region9"]    = poststrat_county["DIVISION"].map(division_map)
poststrat_county["educ_label"] = poststrat_county["educ_category"].map(educ_labels)

# Step 4: merge county covariates
poststrat_county = poststrat_county.merge(
    covars[["county_fips", "county_name", "state_abbr", "co2_per_capita", "dem_share_two_party"]],
    on="county_fips", how="left"
)

# ── Validate ──────────────────────────────────────────────────────────────────
print(f"\nCounty poststrat frame:")
print(f"  Rows:     {len(poststrat_county):,}")
print(f"  Counties: {poststrat_county['county_fips'].nunique():,}")
print(f"  Total weighted adult pop: {poststrat_county['N'].sum():>15,.0f}")
print(f"  Missing co2:      {poststrat_county['co2_per_capita'].isna().sum():,} cells")
print(f"  Missing dem_share:{poststrat_county['dem_share_two_party'].isna().sum():,} cells")
print("\nSample:")
print(poststrat_county.head(8).to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
OUT = "/Users/carmenk/Documents/CSS/Capstone/data/poststrat_county.csv"
poststrat_county.to_csv(OUT, index=False)
print(f"\nSaved → {OUT}")
print(f"Columns: {poststrat_county.columns.tolist()}")

# Poststratification Frame

Build the MRP poststratification frame: joint population counts for every
**state × gender × race (4 cat) × education (4 cat)** cell, using PUMS
person weights (`PWGTP`).

**Race 4-category recode** (matches the CCAM paper):
| ACS rules | Label |
|---|---|
| `HISP > 1` (any Hispanic/Latino, any race) | `Hispanic` |
| `HISP == 1` & `RAC1P == 1` | `White` (non-Hispanic) |
| `HISP == 1` & `RAC1P == 2` | `Black` (non-Hispanic) |
| all remaining non-Hispanic | `Other` |

Scope: **adults 18+** only (matching survey population).

In [13]:
# ── Reload all four person files with HISP and AGEP ───────────────────────────
import pandas as pd
import numpy as np

DATA_DIR = "/Users/carmenk/Documents/CSS/Capstone/acs2024_csv_5year/"
POSTSTRAT_COLS = ["PWGTP", "STATE", "DIVISION", "SEX", "RAC1P", "HISP", "SCHL", "AGEP"]

parts2 = []
for suffix in ["a", "b", "c", "d"]:
    path = f"{DATA_DIR}psam_pus{suffix}.csv"
    chunk = pd.read_csv(path, usecols=POSTSTRAT_COLS, low_memory=False)
    parts2.append(chunk)
    print(f"  psam_pus{suffix}.csv  → {len(chunk):,} rows")

acs2 = pd.concat(parts2, ignore_index=True)
print(f"\nTotal: {len(acs2):,} rows | States: {acs2['STATE'].nunique()}")

  psam_pusa.csv  → 4,748,656 rows
  psam_pusb.csv  → 3,468,017 rows
  psam_pusc.csv  → 3,757,739 rows


KeyboardInterrupt: 

In [ ]:
# ── Recode: adults, gender, race4, educ_category ──────────────────────────────

# Adults only — survey population is 18+
acs2 = acs2[acs2["AGEP"] >= 18].copy()
print(f"Adults (18+): {len(acs2):,}")

# gender
acs2["gender"] = acs2["SEX"].map({1: "Male", 2: "Female"})

# race4 — vectorized, using HISP + RAC1P
# HISP: 1 = Not Hispanic/Latino, 2–24 = Hispanic/Latino (various origins)
hisp_num = pd.to_numeric(acs2["HISP"], errors="coerce")
is_hisp  = hisp_num > 1
is_white = (~is_hisp) & (acs2["RAC1P"] == 1)
is_black = (~is_hisp) & (acs2["RAC1P"] == 2)

acs2["race4"] = np.select(
    [is_hisp,     is_white,  is_black],
    ["Hispanic",  "White",   "Black"],
    default="Other"
)

# educ_category — pd.cut over SCHL codes 1–24
# (0, 15] → 1 Less than HS | (15, 17] → 2 HS | (17, 20] → 3 Some college | (20, 24] → 4 BA+
acs2["educ_category"] = pd.cut(
    pd.to_numeric(acs2["SCHL"], errors="coerce"),
    bins=[0, 15, 17, 20, 24],
    labels=[1, 2, 3, 4]
).astype("Int8")

educ_labels = {1: "Less than HS", 2: "High school",
               3: "Some college",  4: "Bachelor's or higher"}
division_map = {
    1: "New England",    2: "Mid-Atlantic",      3: "E. North Central",
    4: "W. North Central", 5: "South Atlantic",  6: "E. South Central",
    7: "W. South Central", 8: "Mountain",        9: "Pacific",
}

print("\nRace4 distribution (adults, unweighted):")
print(acs2["race4"].value_counts(normalize=True).mul(100).round(2).to_string())
print("\nEduc distribution (adults, unweighted):")
print(acs2["educ_category"].map(educ_labels)
      .value_counts(normalize=True).mul(100).round(2).to_string())

In [ ]:
# ── Build and validate poststratification frame ───────────────────────────────

valid = acs2.dropna(subset=["gender", "race4", "educ_category"])
print(f"Rows dropped (missing demographics): {len(acs2) - len(valid):,}")

# Sum PWGTP within each state × gender × race4 × educ cell
poststrat = (
    valid
    .groupby(["STATE", "DIVISION", "gender", "race4", "educ_category"],
             observed=True)["PWGTP"]
    .sum()
    .reset_index()
    .rename(columns={"PWGTP": "N", "STATE": "state", "DIVISION": "division"})
)

# Add human-readable labels
poststrat["region9"]    = poststrat["division"].map(division_map)
poststrat["educ_label"] = poststrat["educ_category"].map(educ_labels)

# ── Validation ────────────────────────────────────────────────────────────────
max_cells = 51 * 2 * 4 * 4   # states × gender × race4 × educ
print(f"\nPoststrat frame: {len(poststrat):,} rows  (max possible: {max_cells})")
print(f"  States covered: {poststrat['state'].nunique()}")
print(f"  Total weighted adult population: {poststrat['N'].sum():>15,.0f}")

# Check completeness: every state should have all 2×4×4 = 32 cells
cell_counts = poststrat.groupby("state").size()
incomplete = cell_counts[cell_counts < 32]
if incomplete.empty:
    print("  All 51 states have all 32 demographic cells ✓")
else:
    print(f"  States with fewer than 32 cells (sparse): {incomplete.to_dict()}")

print("\nSample rows:")
print(poststrat.head(12).to_string(index=False))

# ── Save ──────────────────────────────────────────────────────────────────────
OUT_PATH = "/Users/carmenk/Documents/CSS/Capstone/poststrat_frame.csv"
poststrat.to_csv(OUT_PATH, index=False)
print(f"\nSaved → {OUT_PATH}")
print(f"Columns: {poststrat.columns.tolist()}")